# Readme

#### Here we estimate the number of events
*   We integrate $\frac{d^2\sigma}{dE_Xdcosθ_X}$ for total cross section $σ$
*   We sample from $P(E_X,cos\theta_X) = \frac{1}{\sigma}\frac{d^2\sigma}{dE_Xdcosθ_X}$ using importance sampling
*   We take the effective acceptance ratio from the samples
*   We report number of events as: $N = \ell\cdot BR_{X\rightarrow F}\cdot E[A] \cdot \sigma$



*   There are 2 general approaches: working with $A(EX,cosX)$ or sampling for $E[A(EX,cosX]$.

*   It's of note that the sampling needs to be able to be done with respect to free parameters: mX on (0,2)GeV , s_tot, and xq

*   $N = \ell\cdot BR_{X\rightarrow F}\cdot E[A] \cdot \sigma$
*   $\frac{dN}{dE_Xdcos\theta_x} = ℓ \cdot BR_{X\rightarrow F}\cdot A(E_X,cos\theta_x) \cdot \frac{d^2\sigma}{dE_Xdcosθ_X} $

# imports

In [ ]:
import photoproductionmodel as pm
import branching_ratios as br
import numpy as np
import scipy
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
from google.colab import output
output.enable_custom_widget_manager()


mp = 0.9382720813
e = 0.303
# gauge coupling charges
gauge_couplings = {"A'":np.array([2/3,-1/3,-1/3,-1,-1,0,0,0]),
                   "B-L":np.array([1/3,1/3,1/3,-1,-1,-1,-1,-1]),
                   "B":np.array([1/3,1/3,1/3,-e**2/(4*np.pi)**2,-e**2/(4*np.pi)**2,0,0,0]),
                   "Chargephobic":np.array([-1/(3*np.sqrt(2)),np.sqrt(2)/3,np.sqrt(2)/3,0,0,-1/np.sqrt(2),-1/np.sqrt(2),-1/np.sqrt(2)])}


#photoproduction model params CONVERGED
params = np.array([11.81336049,
                   3.92112205,
                   0.77877409,
                   0.68472406,
                   0.89693757,
                   0.80841249,
                   0.61509084,
                   0.69624785,
                   -4.21703614,
                   1.90289369,
                   8.81722149,
                   3.12168375,
                   1.1120275,
                   9.47975335,
                   0.92746299,
                   0.80066329,
                   0.80370305,
                   0.56739554,
                   1.74087675,
                   2.21138913,
                   5.52013592,
                   1.06419093,
                   0.99383077])

## Rotate 3d plots install (optional)

In [ ]:
# !pip install ipympl
'''
1. run the install above
2. restart kernel
3. do not run the install again
4. run %matplotlib widget
5. continue
'''

In [ ]:
# %matplotlib widget

# Acceptance function

In [ ]:
def A(cosX,EX,L0,alphaX,mX,x):

  gamma = EX/mX #GeV/GeV unitless
  c_Tau = br.get_decay_length(mX,alphaX,x) #[units of length same as L0]

  return np.exp(-L0/(gamma*c_Tau*cosX)) #unitless

'''
* L0 is going to be given in some metric units of length like [m]
* the denominator in exponent must match those units
* cosX has no units,
  c_Tau has units determined in br script,
  gamma is fundamentally unitless: EX is GeVs so the denominator must also be in GeVs,
  c_Tau and L0 must have the same units
'''


# Integrating total cross section and acceptance

### submethods

In [ ]:
def kinematically_allowed_region_EX_cosX(s_tot,mX,resolution):
  '''
  s_tot: float experimental c.o.m. energy squared
  mX: float mass of X boson
  resolution: int resolution of rectangular point cloud
  returns EX_allowed, cosX_allowed numpy arrays of allowed coordinates in (EX,cosX)
  '''
  E_beam = (s_tot - mp**2) / (2 * mp)

  # uniform point cloud in rectangular region
  cosX = np.linspace(-1,1,resolution)
  EX = np.linspace(mX + 1e-6,E_beam,resolution)

  EX_grid, cosX_grid = np.meshgrid(EX,cosX)

  # point cloud limited to kinematically allowed region (1D objs: EX_allowed, cosX_allowed)
  qX = np.sqrt(EX_grid**2 - mX**2)
  nu = (mp * EX_grid - 0.5 * mX**2) / (mp - EX_grid + qX * cosX_grid)

  validity_mask = (nu>EX) & (nu<E_beam)
  EX_allowed, cosX_allowed = EX_grid[validity_mask] , cosX_grid[validity_mask]

  return EX_allowed, cosX_allowed

### plotting regions and integrands

In [ ]:
def plot_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq)

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, dsig,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('allowed region')
  plt.show()

In [ ]:
def plot_A_times_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution):
  '''
  3d point cloud of dsigX_dEX_dcosX * A(EX,cosX)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  dsig = np.zeros(len(EX_allowed))
  for i in range(len(EX_allowed)):
    dsig[i] = pm.dsigX_dEX_dcosX(s_tot,EX_allowed[i],cosX_allowed[i],params,mX,xq) #put A here

  # plotting point cloud
  fig = plt.figure()
  point_cloud_ax = plt.axes(projection='3d')
  point_cloud_ax.scatter(EX_allowed, cosX_allowed, dsig,marker='v',s=0.01)
  point_cloud_ax.set_xlabel('EX')
  point_cloud_ax.set_ylabel('cosX')
  point_cloud_ax.set_zlabel('dsigX_dEX_dcosX')
  point_cloud_ax.set_title('allowed region')
  plt.show()

In [ ]:
def plot_allowed_region(s_tot,mX,resolution):
  '''
  plots point cloud of allowed region
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  fig = plt.figure()
  plt.scatter(EX_allowed,cosX_allowed,s=0.1)
  plt.title('valid threshold')
  plt.xlabel('EX')
  plt.ylabel('cosX')

In [ ]:
def spline_region(s_tot,mX,resolution,plot = False):
  '''
  splines for dblquad integration
  plots point cloud allowed region and splines of upper and lower EX(cosX)
  return sequence of CubicSpline objects (spline_EX_upper, spline_EX_lower)
  '''
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)

  # interpolating region boundary functions of EX wrt cosX
  cosX_unique = np.unique(cosX_allowed)

  EX_upper = np.zeros(len(cosX_unique))
  EX_lower = np.zeros(len(cosX_unique)) # arrays of upper and lower values of EX for the set of unique cosXs in our point cloud

  for i in range(len(cosX_unique)):
    key = np.where(cosX_allowed == cosX_unique[i],True,False)
    EX_upper[i]= np.max(EX_allowed[key])
    EX_lower[i] = np.min(EX_allowed[key])

  spline_EX_upper = scipy.interpolate.CubicSpline(cosX_unique,EX_upper,extrapolate=False)
  spline_EX_lower = scipy.interpolate.CubicSpline(cosX_unique,EX_lower,extrapolate=False)

  if plot:
    fig = plt.figure()
    plt.scatter(EX_allowed,cosX_allowed,s=0.1)
    cosX_range = np.linspace(np.min(cosX_unique),np.max(cosX_unique),resolution)
    plt.plot(spline_EX_upper(cosX_range),cosX_range,color='g')
    plt.plot(spline_EX_lower(cosX_range),cosX_range,color='r')
    plt.title('valid threshold, splines')
    plt.xlabel('EX')
    plt.ylabel('cosX')
    plt.show()

  return spline_EX_upper, spline_EX_lower

### computing integrals

In [ ]:
def dsig_integrand(EX,cosX,s_tot,params,mX,xq):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq)

def dsig_times_acceptance_integrand(EX,cosX,s_tot,params,mX,xq):
  '''
  integrand formatted for dblquad integrating
  '''
  return pm.dsigX_dEX_dcosX(s_tot,EX,cosX,params,mX,xq) # multiply by A(EX,cosX,L0,c_Tau,mX) here

def integrate_dsig(s_tot,params,mX,xq,resolution):
  '''
  integrates d_sigX_dEX_dcosX with scipy.dblquad
  return float value of integral
  '''
  # region boundaries
  EX_allowed, cosX_allowed = kinematically_allowed_region_EX_cosX(s_tot,mX,resolution)
  cosX_lower_boundary = np.min(cosX_allowed)
  cosX_upper_boundary = np.max(cosX_allowed)
  EX_upper_boundary_func , EX_lower_boundary_func = spline_region(s_tot,mX,resolution)

  # integration
  result , error = scipy.integrate.dblquad(
    dsig_integrand,
    cosX_lower_boundary,
    cosX_upper_boundary,
    EX_lower_boundary_func,
    EX_upper_boundary_func,
    args=(s_tot,params,mX,xq)
    )

  return result

# Scratch

# Notes

In [ ]:
# importance sampling will not be the route I go with because the proposal would have to change with mX, s_tot, and xq
# I have the mcmc independence sampling working, it just doesn't produce a lot of samples for higher masses, see Branching_Ratios.ipynb {Sampling Kinematic Variables}

# Integrating our dsig. let's try quadrature with scipy.quad and see what fails

##### integrating

* I am going to pass $\frac{d^2\sigma}{dE_Xdcosθ_X}$ into

```
scipy.integrate.dblquad(func, a, b, gfun, hfun)
```
* To do this I need to define the region of integration, which is demonstrated in

```
plot_valid_threshold(s_tot,params,mX,xq,resolution)
```

* To do that I need to define the function of the kinematically allowed value of cosX w.r.t to EX or vice versa, which I can do by interpolating the plotted data


# Execute code

### set parameters

In [ ]:
s_tot = 76
mX = 2
x = gauge_couplings["Chargephobic"]
xq = x[0:3]
resolution = 100

### $\sigma_{total}$ and plots

In [ ]:
result = integrate_dsig(s_tot,params,mX,xq,resolution)
print('sig_tot = ' + str(f"{result:.16e}"))
plot_dsigX_dEX_dcosX(s_tot,params,mX,xq,resolution)
spline_region(s_tot,mX,resolution,True)

# Note

* Splines look weird at low resolution for low masses. m=0.1, res=1000 seems valid
* Probably fixable by automatically setting the point resolution in the valid threshold so that EX is sufficiently divided
* e.g. (m = 0.1 , res = 100) is visibly bad boundary spline, whereas ( m = 2 res = 100) does a lot better even with less points